# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
print("Dataset Record Sets:")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']} (name: {record_set.get('name', '')})")
    print("  Fields:")
    for field in record_set['field']:
        if isinstance(field, dict):
            field_id = field.get('@id', field)
            field_name = field.get('name', '')
        else:
            field_id = field
            field_name = ''
        print(f"    - {field_id} (name: {field_name})")
    print()

## 3. Data Extraction
Load data from the main record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
#---
# Replace the below list of record set @ids by what you found from the cell above. For the example, we assume only one main table.
# In FAIR^2, main record set is typically something like:
# '@id': 'https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0-records'
# If there are multiple, add their @ids to the list below.
record_sets = [record_set['@id'] for record_set in metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in record set {record_set_id}: {df.columns.tolist()}")
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming numeric distributions, and grouping.

In [ ]:
#---
# Example: Use the main table for EDA. We choose the first loaded record set for demo
record_set_id = record_sets[0]
df = dataframes[record_set_id]

# Print non-object column types (numeric)
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

# Choose a numeric field for filtering and normalization; edit here based on columns shown above
if numeric_cols:
    numeric_field = numeric_cols[0]
else:
    # If none autodetected, specify manually (e.g., 'MolecularWeight')
    numeric_field = df.columns[0]

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head(3))

# Normalize
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

# Group by a categorical field if available (such as 'Sample' or 'Experiment')
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
group_field = cat_cols[0] if cat_cols else None
if group_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head(3))

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Visualize relation to group if available
    if group_field:
        plt.figure(figsize=(9,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, and exploratory analysis of the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset with `mlcroissant`.

- We loaded Croissant metadata, identified record sets and fields by their `@id`,
- Extracted and analyzed main tabular data,
- Applied EDA to numeric columns (e.g., filtering, normalization, and summary grouping),
- Generated visualizations for quick insights.

You can now continue advanced analyses or modeling using these data structures and their `@id`-referenced fields for robust reproducibility.